# 08 — Mixture of Experts — Solution

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from src.models.moe import MoELayer, MoEBlock
from src.models.feedforward import FeedForward

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Part B — Simple Top-K Router

In [ ]:
class SimpleTopKRouter(nn.Module):
    def __init__(self, d_model, num_experts, top_k=2):
        super().__init__()
        self.top_k = top_k
        self.num_experts = num_experts
        self.gate = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x):
        logits = self.gate(x)                                   # (B, S, E)
        top_k_logits, indices = torch.topk(logits, self.top_k, dim=-1)  # (B, S, k)
        weights = F.softmax(top_k_logits, dim=-1)               # normalise among chosen
        return weights, indices, logits


router = SimpleTopKRouter(64, 8, top_k=2)
x_t = torch.randn(2, 10, 64)
weights, indices, logits = router(x_t)
print(f'Weights: {weights.shape}   indices: {indices.shape}   sum: {weights.sum(-1).mean():.4f}')

## Part C — Load-Balancing Loss

In [ ]:
def load_balancing_loss(router_logits, expert_indices, num_experts):
    batch, seq_len, top_k = expert_indices.shape
    total_tokens = batch * seq_len

    # f_i: fraction of tokens routed to expert i
    counts = torch.zeros(num_experts, device=router_logits.device)
    for e in range(num_experts):
        counts[e] = (expert_indices == e).float().sum()
    f = counts / (total_tokens * top_k)   # normalise by total expert-token assignments

    # P_i: mean router probability for expert i
    probs = F.softmax(router_logits, dim=-1)   # (B, S, E)
    P = probs.mean(dim=[0, 1])                 # (E,)

    return num_experts * (f * P).sum()


# Visualise distribution with random weights
router_vis = SimpleTopKRouter(64, 8, top_k=2)
x_vis = torch.randn(4, 20, 64)
w_vis, idx_vis, log_vis = router_vis(x_vis)

counts = torch.zeros(8)
for e in range(8): counts[e] = (idx_vis == e).float().sum()

plt.figure(figsize=(7, 3))
plt.bar(range(8), counts.detach().numpy())
plt.xlabel('Expert ID'); plt.ylabel('Tokens assigned')
plt.title('Expert Load (random weights — roughly uniform)')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

aux = load_balancing_loss(log_vis, idx_vis, 8)
print(f'\nAuxiliary loss: {aux.item():.4f}  (1.0 = perfectly balanced)')

## Part D — Full MoE Layer

In [ ]:
moe = MoELayer(d_model=128, num_experts=8, top_k=2)
x = torch.randn(2, 16, 128)
out, aux = moe(x)
print(f'MoELayer: {out.shape}  aux_loss={aux.item():.4f}')

moe_block = MoEBlock(d_model=128, n_heads=4, num_experts=8, top_k=2)
out2, aux2 = moe_block(x)
print(f'MoEBlock: {out2.shape}  aux_loss={aux2.item():.4f}')

## Part E — Parameter vs Compute

In [ ]:
d_model, d_ff, num_experts, top_k = 512, 2048, 8, 2
dense_params = sum(p.numel() for p in FeedForward(d_model, d_ff).parameters())
moe_total  = dense_params * num_experts
moe_active = dense_params * top_k

print(f'Dense FFN     : {dense_params:,} params  (all active)')
print(f'MoE {num_experts}×FFN  : {moe_total:,} params  ({moe_active:,} active = same compute as dense×{top_k})')
print(f'\nParameter scaling: {moe_total/dense_params:.0f}x  |  Compute scaling: {moe_active/dense_params:.0f}x')

# Mixtral reference
print('\nMixtral-8x7B context:')
print('  8 experts, top-2 → 47B total params, 13B active per token')
print('  Same compute as a 13B dense model, but quality of a ~47B model')